# 🔁 RippleNet: Intention Prediction via EEG/HRV
Simulated LSTM model using PyTorch.


In [ ]:
# Install dependencies (if not already installed)
!pip install -q torch pandas matplotlib scikit-learn seaborn

In [ ]:
# Import packages
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

In [ ]:
# Load simulated dataset
url = 'https://raw.githubusercontent.com/tententch/open-ripple-data/main/sim_ripple_dataset.csv'
df = pd.read_csv(url)
df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df.head()

In [ ]:
# Prepare data
features = df[["EEG_Coherence", "HRV_Entropy"]].values
target = df["Intention_Strength"].values
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Create sequences
time_steps = 10
X_seq, y_seq = [], []
for i in range(time_steps, len(features_scaled)):
    X_seq.append(features_scaled[i-time_steps:i])
    y_seq.append(target[i])
X_tensor = torch.tensor(X_seq, dtype=torch.float32)
y_tensor = torch.tensor(y_seq, dtype=torch.float32).view(-1, 1)

In [ ]:
# Train/test split
train_size = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:train_size], X_tensor[train_size:]
y_train, y_test = y_tensor[:train_size], y_tensor[train_size:]
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)

In [ ]:
# Define RippleNet LSTM model
class RippleNet(nn.Module):
    def __init__(self):
        super(RippleNet, self).__init__()
        self.lstm1 = nn.LSTM(input_size=2, hidden_size=64, batch_first=True)
        self.dropout1 = nn.Dropout(0.2)
        self.lstm2 = nn.LSTM(input_size=64, hidden_size=32, batch_first=True)
        self.dropout2 = nn.Dropout(0.2)
        self.fc = nn.Linear(32, 1)
    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.dropout1(out)
        out, _ = self.lstm2(out)
        out = self.dropout2(out)
        return self.fc(out[:, -1, :])

In [ ]:
# Train model
model = RippleNet()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(30):
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
    if epoch % 5 == 0:
        print(f"Epoch {epoch} - Loss: {loss.item():.4f}")

In [ ]:
# Evaluate
model.eval()
with torch.no_grad():
    y_pred = model(X_test).numpy().flatten()
    y_true = y_test.numpy().flatten()
    mse = mean_squared_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"MSE: {mse:.4f}, R²: {r2:.4f}")

# Plot
plt.figure(figsize=(10,4))
plt.plot(y_true, label='True')
plt.plot(y_pred, label='Predicted', alpha=0.7)
plt.legend()
plt.title("RippleNet Intention Prediction")
plt.tight_layout()
plt.show()